In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA retail")

silver = spark.table("workspace.retail.silver_online_retail")
silver_count = silver.count()
print("Silver rows (fact must match):", f"{silver_count:,}")

In [0]:
real_customers = (
    silver.filter(F.col("customer_id") != -1)
    .groupBy("customer_id")
    .agg(F.min(F.to_date("invoice_date")).alias("first_purchase_date"),
         F.max(F.to_date("invoice_date")).alias("last_purchase_date"))
    .withColumn("customer_key", F.row_number().over(Window.orderBy("customer_id")))
    .withColumn("is_unknown", F.lit(False))
)

unknown_row = spark.createDataFrame(
    [(-1, -1, None, None, True)],
    "customer_key int, customer_id long, first_purchase_date date, last_purchase_date date, is_unknown boolean",
)

dim_customer = real_customers.select(
    "customer_key", "customer_id", "first_purchase_date", "last_purchase_date", "is_unknown"
).unionByName(unknown_row)

dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail.dim_customer")

In [0]:
dim_country = (
    silver.select("country").distinct()
    .withColumn("country_key", F.row_number().over(Window.orderBy("country")))
    .select("country_key", "country")
)

dim_country.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail.dim_country")

display(dim_country)

In [0]:
desc_counts = (
    silver.filter(F.col("description").isNotNull() & (F.col("description") != ""))
    .groupBy("stock_code", "description")
    .agg(F.count("*").alias("n"), F.max("invoice_date").alias("last_seen"))
)

w = Window.partitionBy("stock_code").orderBy(F.desc("n"), F.desc("last_seen"), "description")
best_desc = (
    desc_counts.withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .select("stock_code", "description")
)

dim_product = (
    silver.select("stock_code").distinct()
    .join(best_desc, "stock_code", "left")
    .fillna({"description": "UNKNOWN"})
    .withColumn("product_key", F.row_number().over(Window.orderBy("stock_code")))
    .select("product_key", "stock_code", "description")
)

dim_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail.dim_product")

print("Products:", dim_product.count())

In [0]:
bounds = silver.agg(F.min(F.to_date("invoice_date")).alias("lo"),
                    F.max(F.to_date("invoice_date")).alias("hi")).first()

dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{bounds.lo}'), to_date('{bounds.hi}'), interval 1 day)) AS full_date")
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("year_month", F.date_format("full_date", "yyyy-MM"))
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))
)

dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail.dim_date")

In [0]:
fact_sales = (
    silver
    .join(spark.table("dim_customer").select("customer_key", "customer_id"), "customer_id", "left")
    .join(spark.table("dim_product").select("product_key", "stock_code"), "stock_code", "left")
    .join(spark.table("dim_country"), "country", "left")
    .withColumn("date_key", F.date_format("invoice_date", "yyyyMMdd").cast("int"))
    .select(
        F.col("source_row_id").alias("sales_line_id"),
        "invoice", "invoice_type",
        "date_key", "customer_key", "product_key", "country_key",
        "invoice_date", "quantity", "price", "line_amount",
        "is_possible_duplicate", "dup_seq",
        "source_file",
    )
)

fact_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail.fact_sales")

In [0]:
fact = spark.table("fact_sales")
dc, dp, dco, dd = (spark.table(t) for t in ["dim_customer", "dim_product", "dim_country", "dim_date"])

# 1. Row count: no rows lost or multiplied by joins
f = fact.count()
print(f"Fact: {f:,} | Silver: {silver_count:,}")
assert f == silver_count, "Join dropped or duplicated rows!"

# 2. Grain is unique: one row per sales_line_id
assert fact.select("sales_line_id").distinct().count() == f, "Grain violated"

# 3. No orphan keys
for k in ["customer_key", "product_key", "country_key", "date_key"]:
    assert fact.filter(F.col(k).isNull()).count() == 0, f"NULL {k}"
assert fact.join(dd, "date_key", "left_anti").count() == 0, "date_key not in dim_date"

# 4. Dimension keys are unique
for df, k in [(dc, "customer_key"), (dp, "product_key"), (dco, "country_key"), (dd, "date_key")]:
    assert df.select(k).distinct().count() == df.count(), f"Duplicate {k}"

# 5. Money ties out to Silver
fact_total = fact.agg(F.round(F.sum("line_amount"), 2)).first()[0]
silver_total = silver.agg(F.round(F.sum("line_amount"), 2)).first()[0]
print("Fact total:", fact_total, "| Silver total:", silver_total)
assert fact_total == silver_total

# 6. Unknown customer is kept
unk = fact.filter("customer_key = -1").agg(F.count("*").alias("rows"),
                                           F.round(F.sum("line_amount"), 2).alias("revenue"))
display(unk)

print("All model checks passed")